In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

CURRENT_DIR = os.getcwd()
POSSIBLE_RESULTS_DIRS = [
    os.path.abspath(os.path.join(CURRENT_DIR, "..", "..", "MI BCI", "results")),
    os.path.abspath(os.path.join(CURRENT_DIR, "..", "results")),
    os.path.abspath(os.path.join(CURRENT_DIR, "results")),
]

RESULTS_DIR = None
for p in POSSIBLE_RESULTS_DIRS:
    if os.path.exists(p) and len(glob.glob(os.path.join(p, "*.csv"))) > 0:
        RESULTS_DIR = p
        break

if RESULTS_DIR is None:
    raise FileNotFoundError("Could not find the results directory! Please verify paths.")

print(f"✅ Found Results Directory: {RESULTS_DIR}")

In [2]:
def compute_cohens_d(x, y):
    """
    Computes Cohen's d for paired samples:
    d = mean(x - y) / std(x - y)
    """
    diff = np.array(x) - np.array(y)
    n = len(diff)
    if n < 2:
        return 0.0
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-9)

MODELS = ['eegnet', 'cvt', 'vit_pe']
DATASETS = ['Raw', 'Filtered_8_30Hz', 'Filtered_8_13Hz']
METRICS = ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']

data_store = {}

for model in MODELS:
    for ds in DATASETS:
        filename = f"{model}_results_{ds}.csv"
        filepath = os.path.join(RESULTS_DIR, filename)
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Missing file: {filepath}")
        
        df = pd.read_csv(filepath)
        df.columns = [c.lower() for c in df.columns]
        data_store[(model, ds)] = df

print("✅ All 9 fold result CSV files loaded successfully!")

✅ All 9 fold result CSV files loaded successfully!


In [3]:
summary_records = []

for ds in DATASETS:
    for model in MODELS:
        df = data_store[(model, ds)]
        rec = {
            'Dataset': ds,
            'Model': model.upper().replace('_', '-')
        }
        for metric in METRICS:
            if metric in df.columns:
                mean_val = df[metric].mean() * 100
                std_val = df[metric].std() * 100
                rec[f"{metric.upper()}_Mean"] = mean_val
                rec[f"{metric.upper()}_SD"] = std_val
                rec[f"{metric.upper()} (Mean±SD)"] = f"{mean_val:.2f} ± {std_val:.2f}%"
        summary_records.append(rec)

summary_df = pd.DataFrame(summary_records)
print("--- SUMMARY PERFORMANCE TABLE (Across 5 Folds) ---")
display_cols = ['Dataset', 'Model'] + [f"{m.upper()} (Mean±SD)" for m in METRICS]
display_df = summary_df[display_cols]
print(display_df.to_string(index=False))

summary_df.to_csv("summary_fold_performance.csv", index=False)
print("\n✅ Saved to summary_fold_performance.csv")

--- SUMMARY PERFORMANCE TABLE (Across 5 Folds) ---
        Dataset  Model ACCURACY (Mean±SD)  F1 (Mean±SD) PRECISION (Mean±SD) RECALL (Mean±SD) ROC_AUC (Mean±SD)
            Raw EEGNET      86.26 ± 4.99% 86.17 ± 5.61%       85.76 ± 2.90%    86.82 ± 8.77%     93.42 ± 4.33%
            Raw    CVT      76.23 ± 3.69% 77.22 ± 2.53%       74.87 ± 6.20%    80.31 ± 4.93%     85.11 ± 4.00%
            Raw VIT-PE      67.39 ± 3.42% 67.51 ± 2.54%       67.53 ± 4.30%    67.60 ± 1.98%     73.99 ± 3.76%
Filtered_8_30Hz EEGNET      84.24 ± 4.12% 83.79 ± 4.32%       86.21 ± 4.26%    81.52 ± 4.65%     91.58 ± 4.46%
Filtered_8_30Hz    CVT      77.26 ± 2.25% 76.85 ± 2.35%       78.27 ± 2.54%    75.53 ± 2.87%     85.55 ± 2.57%
Filtered_8_30Hz VIT-PE      73.24 ± 2.03% 72.68 ± 2.38%       74.21 ± 1.92%    71.27 ± 3.42%     81.42 ± 2.27%
Filtered_8_13Hz EEGNET      84.79 ± 3.01% 84.62 ± 3.24%       85.56 ± 3.67%    83.86 ± 4.94%     92.46 ± 3.17%
Filtered_8_13Hz    CVT      78.33 ± 3.91% 78.32 ± 4.07%      

In [4]:
stat_comparisons = []
PAIRS = [
    ('cvt', 'eegnet', 'CvT vs EEGNet'),
    ('vit_pe', 'eegnet', 'ViT-PE vs EEGNet'),
    ('vit_pe', 'cvt', 'ViT-PE vs CvT')
]

for ds in DATASETS:
    for m1, m2, label in PAIRS:
        df1 = data_store[(m1, ds)]
        df2 = data_store[(m2, ds)]
        
        df1 = df1.sort_values(by='fold').reset_index(drop=True)
        df2 = df2.sort_values(by='fold').reset_index(drop=True)
        
        for metric in METRICS:
            if metric in df1.columns and metric in df2.columns:
                scores1 = df1[metric].values
                scores2 = df2[metric].values
                
                # 1. Paired Student's t-test
                t_stat, p_val_ttest = stats.ttest_rel(scores1, scores2)
                
                # 2. Wilcoxon signed-rank test
                try:
                    w_stat, p_val_wilcoxon = stats.wilcoxon(scores1, scores2)
                except Exception:
                    p_val_wilcoxon = 1.0
                    w_stat = 0.0
                
                # 3. Cohen's d (Effect Size)
                d = compute_cohens_d(scores1, scores2)
                
                if abs(d) >= 0.8:
                    d_interp = "Large"
                elif abs(d) >= 0.5:
                    d_interp = "Medium"
                elif abs(d) >= 0.2:
                    d_interp = "Small"
                else:
                    d_interp = "Negligible"
                
                stat_comparisons.append({
                    'Dataset': ds,
                    'Comparison': label,
                    'Metric': metric.upper(),
                    'Mean_Diff (%)': (np.mean(scores1) - np.mean(scores2)) * 100,
                    't_stat': round(t_stat, 4),
                    'p_val (Paired t-test)': f"{p_val_ttest:.4e}" if p_val_ttest < 0.001 else f"{p_val_ttest:.4f}",
                    'p_val (Wilcoxon)': f"{p_val_wilcoxon:.4e}" if p_val_wilcoxon < 0.001 else f"{p_val_wilcoxon:.4f}",
                    "Cohen's d": round(d, 4),
                    'Effect Size': d_interp,
                    'Significant (p<0.05)': 'Yes' if p_val_wilcoxon < 0.05 or p_val_ttest < 0.05 else 'No'
                })

stat_df = pd.DataFrame(stat_comparisons)

stat_df.to_csv("statistical_significance_results.csv", index=False)
print("✅ Saved statistical significance results to: statistical_significance_results.csv")

✅ Saved statistical significance results to: statistical_significance_results.csv


In [5]:
key_stats = stat_df[stat_df['Metric'].isin(['ACCURACY', 'F1'])][
    ['Dataset', 'Comparison', 'Metric', 'Mean_Diff (%)', 'p_val (Paired t-test)', 'p_val (Wilcoxon)', "Cohen's d", 'Effect Size']
]

print("\n" + "="*85)
print("🎯 STATISTICAL COMPARISON SUMMARY TABLE FOR REVIEWER RESPONSE")
print("="*85)
print(key_stats.to_markdown(index=False))


🎯 STATISTICAL COMPARISON SUMMARY TABLE FOR REVIEWER RESPONSE
| Dataset         | Comparison       | Metric   |   Mean_Diff (%) |   p_val (Paired t-test) |   p_val (Wilcoxon) |   Cohen's d | Effect Size   |
|:----------------|:-----------------|:---------|----------------:|------------------------:|-------------------:|------------:|:--------------|
| Raw             | CvT vs EEGNet    | ACCURACY |       -10.0265  |              0.00048611 |             0.0625 |     -4.6427 | Large         |
| Raw             | CvT vs EEGNet    | F1       |        -8.9535  |              0.0048     |             0.0625 |     -2.5298 | Large         |
| Raw             | ViT-PE vs EEGNet | ACCURACY |       -18.8696  |              7.2432e-05 |             0.0625 |     -7.543  | Large         |
| Raw             | ViT-PE vs EEGNet | F1       |       -18.6669  |              0.00034237 |             0.0625 |     -5.0804 | Large         |
| Raw             | ViT-PE vs CvT    | ACCURACY |        -8.84305 | 